<a href="https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis**

One row represents the performance of one content item for one client on one reporting date.

**Tables used**

- dim_content
- fact_content_daily_performance

**Time window**

I will work with a mid-panel month (March 2026) rather than the final month because the final month should remain isolated from feature development and label construction.

**Lane**

CTR / Engagement Opportunity Scoring (Lane 4)

**Output**

A ranked list of pages whose CTR appears lower than expected for similar search-position tiers.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Lane:", "CTR / Engagement Opportunity Scoring")
print("Month:", "2026-03")

Lane: CTR / Engagement Opportunity Scoring
Month: 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**
- impressions
- clicks
- ctr
- gsc_avg_position
- content_age_days

**Label or proxy**
- expected CTR gap
- optimisation priority score

**Context fields**
- report_date
- client_id
- content_id

**Excluded fields**
- trend_direction
- trend_pct
- is_declining_label

These variables are excluded because they contain information derived from the outcome itself and may introduce leakage.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = [
    "impressions",
    "clicks",
    "ctr",
    "gsc_avg_position",
    "content_age_days",
]

excluded = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

print("Features")
print(features)

print("\nExcluded")
print(excluded)


Features
['impressions', 'clicks', 'ctr', 'gsc_avg_position', 'content_age_days']

Excluded
['trend_direction', 'trend_pct', 'is_declining_label']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following checks verify the grain of the data, the available date range, and the presence of missing values.

In [45]:
!pip install -q datasets huggingface_hub pyarrow pandas

In [46]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:1])

h


In [47]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [48]:
from datasets import load_dataset

dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)

In [49]:
print(dim_content)

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
        num_rows: 519606
    })
})


In [50]:
df = dim_content["train"].to_pandas()

print(df.shape)
df.head()

(519606, 26)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [51]:
#Verify the grain
print("Rows:", len(df))
print("Unique content IDs:", df["content_hash_id"].nunique())

Rows: 519606
Unique content IDs: 519606


In [52]:
#Verify the date span
date_cols = [col for col in df.columns if "date" in col.lower()]
print(date_cols)

['content_created_date', 'content_updated_date', 'keyword_created_date', 'last_optimized_date', 'optimization_eligible_date']


In [53]:
for col in date_cols:
    print("\n", col)

    values = df[col].dropna()

    print("Minimum:", values.min())
    print("Maximum:", values.max())


 content_created_date
Minimum: 2024-10-16
Maximum: 2026-07-06

 content_updated_date
Minimum: 2024-10-28
Maximum: 2026-07-06

 keyword_created_date
Minimum: 2024-10-16
Maximum: 2026-07-02

 last_optimized_date
Minimum: 2026-04-24
Maximum: 2026-07-06

 optimization_eligible_date
Minimum: 2026-06-08
Maximum: 2026-08-20


In [54]:
#Verify missing values
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print(missing.head(10))

optimization_eligible_date    474210
last_optimized_date           474210
provider_used                 369936
backlinks                     267474
word_count                    177768
char_count                    177768
main_intent                   148398
competition_level             144456
cpc                           142622
search_volume                 142622
dtype: int64


In [55]:
#See all available columns
print(df.columns.tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


### Verification results

The following checks were performed to verify the contract.

- The number of unique content identifiers matches the number of rows.
- The content table spans from late 2024 to mid-2026.
- Missing values were identified and examined before feature selection.
- The structure of the data was inspected to ensure that the selected features were available at the decision point.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Historical depth differs across clients.
- Some columns contain large numbers of missing values.
- Search position strongly affects CTR.
- Metadata alone cannot fully explain user behaviour.
- Observed patterns should be treated as decision-support signals rather than causal relationships.

In [56]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.